# KRIOS GIS — Fingrid Grid Capacity Layer Check

This notebook discovers and validates the **Fingrid substation consumption capacity** dataset — the authoritative source for the assignment's "Sähkön kulutuskapasiteetti" (electricity consumption capacity) requirement.

### Layer lineage

| Grid Scope web map | Assignment reference | ArcGIS FeatureServer |
|---|---|---|
| `Sähkön / kulutuskapasiteetti` (group) | "use Sähkön kulutuskapasiteetti layer" | — |
| `Substations` (active layer inside group) | The actual data layer | `Kytkinlaitokset_Fingrid` → `Sähköasemat_liityntäkapasiteetti` |

**Key field:** `Kulutus_25` = consumption headroom (MW) per substation

**Source:** [Grid Scope map](https://karttapalaute.fingrid.fi/?setlanguage=en&?link=3opMB) — Fingrid's interactive map for grid connection capacity estimates.

## Workflow

1. **Chunk 1** -- List all FeatureServer services from the ArcGIS host; confirm Kytkinlaitokset_Fingrid exists.
2. **Chunk 2** -- Drill into the layer, show its schema, field definitions, and a sample of attribute values.
3. **Chunk 2b** -- Download ALL Fingrid substations with geometry, save to ./data/fingrid_substations.geojson.
4. **Chunk 3** -- Extract OSM layers (power lines, substations, power plants, data centers) via Overpass API, save to ./data/.
5. **Chunk 4** -- Combined map: Fingrid substations + all OSM layers on a single GeoLibre widget.


In [1]:
import pandas as pd
import requests

# ArcGIS REST endpoint hosting Fingrid's geospatial data
BASE_URL = "https://services2.arcgis.com/uh3cDCipmuPcmxmx/ArcGIS/rest/services"

# Chunk 1: list all native ArcGIS services and keep FeatureServer items.
root_resp = requests.get(f"{BASE_URL}?f=pjson", timeout=45)
print("Root status:", root_resp.status_code)
root_resp.raise_for_status()
root_json = root_resp.json()

services = root_json.get("services", [])
services_df = pd.DataFrame(services)

if services_df.empty:
    raise RuntimeError("No services returned from ArcGIS root endpoint.")

services_df["service_url"] = (
    BASE_URL + "/" + services_df["name"].astype(str) + "/" + services_df["type"].astype(str)
    )
feature_services_df = services_df[services_df["type"].eq("FeatureServer")].copy()

print("Total services:", len(services_df))
print("FeatureServer count:", len(feature_services_df))
display(feature_services_df[["name", "type", "service_url"]].sort_values("name").reset_index(drop=True))

Root status: 200
Total services: 113
FeatureServer count: 113


,name,type,service_url
0,Asemakaavoitettualue_VESISTOALUEET,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
1,Asuinrakennus,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
2,Biokaasu,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
3,EVO_OYK,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
4,EVO_OYK_KIINTEISTÖTUNNUKSET,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
...,...,...,...
108,survey123_af37f13c356e4d2581a14b302ed42ec9_res...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
109,survey123_d8fbac0dec194234a9b66a751e1c8b8a_form,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
110,survey123_f9551175b74545e8a56f137d280efef9_fie...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...
111,survey123_fa3c5b79a00e48b88cf0e75cbd493ac0_fie...,FeatureServer,https://services2.arcgis.com/uh3cDCipmuPcmxmx/...


In [2]:
# Chunk 2: extract the Fingrid substation connection capacity layer and inspect its schema + sample data.
# Assignment: "Grid capacity & headroom — Fingrid — use Sähkön kulutuskapasiteetti layer"
# Published as: Kytkinlaitokset_Fingrid -> Sähköasemat_liityntäkapasiteetti
# Key field: Kulutus_25 = consumption capacity headroom (MW)

TARGET_SERVICE_NAME = "Kytkinlaitokset_Fingrid"
TARGET_LAYER_HINT = "liityntäkapasiteetti"  # matches Sähköasemat_liityntäkapasiteetti

# Locate the target service
target_service = feature_services_df[feature_services_df["name"] == TARGET_SERVICE_NAME].copy()
if target_service.empty:
    raise RuntimeError(f"Target service not found: {TARGET_SERVICE_NAME}")

service_url = target_service.iloc[0]["service_url"]
service_resp = requests.get(f"{service_url}?f=pjson", timeout=45)
print("Service status:", service_resp.status_code)
service_resp.raise_for_status()
service_json = service_resp.json()

layers = service_json.get("layers", [])
layers_df = pd.DataFrame(layers)
if layers_df.empty:
    raise RuntimeError("No layers found inside target FeatureServer.")

# Find the substation connection capacity layer by name hint
layers_df["name_lc"] = layers_df["name"].astype(str).str.lower()
layer_match = layers_df[layers_df["name_lc"].str.contains(TARGET_LAYER_HINT.lower(), na=False)].copy()
selected_layer = layer_match.iloc[0] if not layer_match.empty else layers_df.iloc[0]

layer_url = f"{service_url}/{int(selected_layer['id'])}"
print("Selected layer:", selected_layer["name"])
print("Layer URL:", layer_url)

# Fetch layer metadata
meta_resp = requests.get(f"{layer_url}?f=pjson", timeout=45)
print("Layer metadata status:", meta_resp.status_code)
meta_resp.raise_for_status()
meta = meta_resp.json()

meta_summary = {
    "name": meta.get("name"),
    "type": meta.get("type"),
    "geometryType": meta.get("geometryType"),
    "objectIdField": meta.get("objectIdField"),
    "displayField": meta.get("displayField"),
    "maxRecordCount": meta.get("maxRecordCount"),
}
print("\nMetadata summary:", meta_summary)

# Show field definitions
fields_df = pd.DataFrame(meta.get("fields", []))
print("\nField count:", len(fields_df))
if not fields_df.empty:
    cols = [c for c in ["name", "alias", "type", "length", "nullable"] if c in fields_df.columns]
    display(fields_df[cols].sort_values("name").reset_index(drop=True))

# Query 10 sample records (no geometry, compact)
query_url = f"{layer_url}/query"
params = {
    "f": "json",
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "resultRecordCount": 10,
}
sample_resp = requests.get(query_url, params=params, timeout=45)
print("\nSample query status:", sample_resp.status_code)
sample_resp.raise_for_status()
sample_json = sample_resp.json()

rows = [f.get("attributes", {}) for f in sample_json.get("features", [])]
sample_df = pd.DataFrame(rows)
print("Sample row count:", len(sample_df))
if not sample_df.empty:
    preferred_cols = [c for c in ["SA", "Tyyppi", "Kulutus_25", "Tuotanto__", "Tuotanto_1", "Tuotanto_2", "X", "Y"] if c in sample_df.columns]
    display(sample_df[preferred_cols] if preferred_cols else sample_df.head(10))

Service status: 200
Selected layer: Sähköasemat_liityntäkapasiteetti
Layer URL: https://services2.arcgis.com/uh3cDCipmuPcmxmx/ArcGIS/rest/services/Kytkinlaitokset_Fingrid/FeatureServer/0
Layer metadata status: 200

Metadata summary: {'name': 'Sähköasemat_liityntäkapasiteetti', 'type': 'Feature Layer', 'geometryType': 'esriGeometryPoint', 'objectIdField': 'FID', 'displayField': 'SA', 'maxRecordCount': 2000}

Field count: 9


,name,alias,type,length,nullable
0,FID,FID,esriFieldTypeOID,NaN,False
1,Kulutus_25,Kulutus_25,esriFieldTypeInteger,NaN,True
2,SA,SA,esriFieldTypeString,254.0,True
3,Tuotanto_1,Tuotanto_1,esriFieldTypeInteger,NaN,True
4,Tuotanto_2,Tuotanto_2,esriFieldTypeInteger,NaN,True
5,Tuotanto__,Tuotanto__,esriFieldTypeInteger,NaN,True
6,Tyyppi,Tyyppi,esriFieldTypeString,254.0,True
7,X,X,esriFieldTypeDouble,NaN,True
8,Y,Y,esriFieldTypeDouble,NaN,True



Sample query status: 200
Sample row count: 10


,SA,Tyyppi,Kulutus_25,Tuotanto__,Tuotanto_1,Tuotanto_2,X,Y
0,AHVENKOSKI 110 kV,110,0,200,200,200,469786.110,6706431.649
1,ALAJÄRVI 110 kV,110,500,60,60,60,356668.873,6992357.131
2,ALAJÄRVI 400 kV,400,500,0,0,0,356668.873,6992357.131
3,ALAPITKÄ 110 kV,110,200,180,180,180,526241.848,7006646.487
4,ALAPITKÄ 400 kV,400,500,380,380,380,526241.848,7006646.487
5,ANTTILA 110 kV,110,0,500,500,500,409988.222,6694581.005
6,ANTTILA 400 kV,400,0,500,500,500,409988.222,6694581.005
7,ARKKUKALLIO 110 kV,110,500,0,240,240,222165.836,6898752.332
8,ARKKUKALLIO 400 kV,400,500,0,440,440,222165.836,6898752.332
9,ESPOO 110 kV,110,0,500,500,500,364250.795,6676726.023


In [ ]:
# Chunk 2b: download ALL Fingrid substations with geometry and save to GeoJSON
# Uses the layer_url from Chunk 2 (must be run first)

from pathlib import Path
import json
import math
import time

OUT = Path("./data")
OUT.mkdir(parents=True, exist_ok=True)

# ArcGIS query: paginate through all records with geometry
def query_arcgis_all(url, where="1=1", out_fields="*", out_sr=4326, page_size=2000):
    """Fetch all features from an ArcGIS FeatureServer layer with pagination."""
    all_features = []
    offset = 0
    while True:
        params = {
            "f": "json",
            "where": where,
            "outFields": out_fields,
            "returnGeometry": "true",
            "outSR": out_sr,
            "resultRecordCount": page_size,
            "resultOffset": offset,
        }
        r = requests.get(f"{layer_url}/query", params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        features = data.get("features", [])
        if not features:
            break
        all_features.extend(features)
        offset += page_size
        time.sleep(0.5)
    return all_features

print("Downloading all Fingrid substations...")
raw_features = query_arcgis_all(layer_url, out_sr=4326)
print(f"Fetched {len(raw_features)} features")

# Convert ArcGIS JSON to GeoJSON
geojson_features = []
for f in raw_features:
    attrs = f.get("attributes", {})
    geom = f.get("geometry")
    if geom and "x" in geom and "y" in geom:
        feature = {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [geom["x"], geom["y"]]},
            "properties": {
                "name": attrs.get("SA", ""),
                "voltage_kv": attrs.get("Tyyppi", None),
                "consumption_mw": attrs.get("Kulutus_25", None),
                "production_1_mw": attrs.get("Tuotanto_1", None),
                "production_2_mw": attrs.get("Tuotanto_2", None),
                "production_total_mw": attrs.get("Tuotanto__", None),
            }
        }
        feature["properties"] = {k: v for k, v in feature["properties"].items() if v is not None}
        geojson_features.append(feature)

fc = {"type": "FeatureCollection", "features": geojson_features}
out_path = OUT / "fingrid_substations.geojson"
out_path.write_text(json.dumps(fc, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(geojson_features)} features to {out_path}")


In [ ]:
# Chunk 3: extract OSM layers via Overpass API (power network, generation, data centers)
# AOI: Helsinki region (~100km radius: 59.7-60.7 lat, 23.9-26.0 lon)

import json
from pathlib import Path
import requests
import time

UA = "KRIOS-GIS-Assignment/1.0 (research)"
OSM_BBOX = "59.7,23.9,60.7,26.0"
PROC = Path("./data")
PROC.mkdir(parents=True, exist_ok=True)

def overpass_to_geojson(query, label):
    """Run Overpass query with retry on 429, convert to GeoJSON FeatureCollection."""
    max_retries = 3
    for attempt in range(max_retries + 1):
        r = requests.post(
            "https://overpass-api.de/api/interpreter",
            data={"data": query},
            headers={"User-Agent": UA},
            timeout=120
        )
        if r.status_code == 429 and attempt < max_retries:
            wait = 10 * (2 ** attempt)  # 10, 20, 40s
            print(f"Rate limited, retrying in {wait}s...")
            time.sleep(wait)
            continue
        r.raise_for_status()
        break
    data = r.json()
    elements = data.get("elements", [])

    features = []
    for el in elements:
        geom = None
        if el["type"] == "node":
            geom = {"type": "Point", "coordinates": [el["lon"], el["lat"]]}
        elif el["type"] == "way" and "geometry" in el:
            coords = [[p["lon"], p["lat"]] for p in el["geometry"]]
            if len(coords) == 1:
                geom = {"type": "Point", "coordinates": coords[0]}
            else:
                geom = {"type": "LineString", "coordinates": coords}
        if geom:
            features.append({
                "type": "Feature",
                "geometry": geom,
                "properties": el.get("tags", {})
            })

    fc = {"type": "FeatureCollection", "features": features}
    path = PROC / f"osm_{label.lower().replace(chr(32), chr(95)).replace(chr(47), chr(95)).replace(chr(40), chr(95)).replace(chr(41), chr(95))}.geojson"
    path.write_text(json.dumps(fc, ensure_ascii=False), encoding="utf-8")
    print(f"{label}: {len(features)} features -> {path}")
    time.sleep(5)
    return fc

# --- 3a. Power lines (110, 220, 400 kV) ---
q = (
    f'[out:json][timeout:60][bbox:{OSM_BBOX}];'
    'way["power"="line"]["voltage"~"400000|220000|110000"];'
    'out geom;'
)
lines = overpass_to_geojson(q, "Power lines (110-400 kV)")

# --- 3b. Substations ---
q = (
    f'[out:json][timeout:60][bbox:{OSM_BBOX}];'
    '('
    '  node["power"="substation"];'
    '  way["power"="substation"];'
    ');'
    'out geom;'
)
substations = overpass_to_geojson(q, "Substations (OSM)")

# --- 3c. Power plants ---
q = (
    f'[out:json][timeout:60][bbox:{OSM_BBOX}];'
    '('
    '  node["power"="plant"];'
    '  way["power"="plant"];'
    '  node["power"="generator"];'
    '  way["power"="generator"];'
    ');'
    'out geom;'
)
plants = overpass_to_geojson(q, "Power plants/generators")

# --- 3d. Data centers ---
q = (
    f'[out:json][timeout:60][bbox:{OSM_BBOX}];'
    '('
    '  node["datacenter"~"yes"];'
    '  way["datacenter"~"yes"];'
    '  node["building"="data_center"];'
    '  way["building"="data_center"];'
    ');'
    'out geom;'
)
datacenters = overpass_to_geojson(q, "Data centers")


In [ ]:
# Chunk 4: combined map -- Fingrid + OSM layers on a single GeoLibre widget

from geolibre import Map

# Load Fingrid GeoJSON
fingrid_path = Path("./data/fingrid_substations.geojson")
fingrid = json.loads(fingrid_path.read_text(encoding="utf-8"))

# Create GeoLibre map centred on Helsinki region
m = Map(
    center=[25.0, 60.2],
    zoom=9,
    basemap="bright",
    layout="embed",
    height="700px",
)

# 1. Fingrid substations (consumption capacity)
m.add_choropleth(
    fingrid,
    column="consumption_mw",
    name="Fingrid Substations (consumption MW)",
    class_count=5,
    colormap="YlOrRd",
    scheme="quantile",
    type="circle",
    radius=10,
    stroke=True,
    popup=["name", "voltage_kv", "consumption_mw"],
)

# 2. OSM power lines
m.add_geojson(
    lines,
    name="Power lines (OSM)",
    color="#e74c3c",
    weight=2,
    popup=["voltage", "name", "operator"],
)

# 3. OSM substations
m.add_geojson(
    substations,
    name="Substations (OSM)",
    color="#3498db",
    radius=6,
    popup=["name", "voltage"],
)

# 4. OSM power plants
m.add_geojson(
    plants,
    name="Power plants (OSM)",
    color="#2ecc71",
    radius=8,
    popup=["name", "plant:source", "generator:source"],
)

# 5. OSM data centers
m.add_geojson(
    datacenters,
    name="Data centers (OSM)",
    color="#9b59b6",
    radius=8,
    popup=["name", "operator"],
)

m
